In [2]:
import pandas as pd
import os

In [3]:
tasks_all = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/Export-Video Mappings.xlsx - Foglio1.csv")
fixations_df = pd.read_csv("outputs/fixations_all_rows.csv")
gaze_df      = pd.read_csv("outputs/gaze_all_rows.csv")
# 1) Prepare task intervals
tasks = tasks_all.rename(columns={
    "Video Name": "Video Name (folder)",
    "User ID": "User ID (file)",
    "Starting time in seconds": "t_start",
    "Finishing time in seconds": "t_end",
}).copy()

tasks["Task Id"] = tasks["Task Id"].astype(str)
fixations_df["Task Id"] = fixations_df["Task Id"].astype(str)
gaze_df["Task Id"]      = gaze_df["Task Id"].astype(str)

# 2) Merge interval bounds onto rows (for ON detection)
fix_m = fixations_df.merge(
    tasks[["Video Name (folder)","User ID (file)","Task Id","t_start","t_end"]],
    on=["Video Name (folder)","User ID (file)","Task Id"], how="left"
)
gaz_m = gaze_df.merge(
    tasks[["Video Name (folder)","User ID (file)","Task Id","t_start","t_end"]],
    on=["Video Name (folder)","User ID (file)","Task Id"], how="left"
)

# 3) Initialize labels
fix_m["label"] = "UNLABELED"
gaz_m["label"] = "UNLABELED"

# 4) ON = inside [start, end]
fix_on = fix_m["TIME"].between(fix_m["t_start"], fix_m["t_end"], inclusive="both")
gaz_on = gaz_m["TIME"].between(gaz_m["t_start"], gaz_m["t_end"], inclusive="both")
fix_m.loc[fix_on, "label"] = "ON"
gaz_m.loc[gaz_on, "label"] = "ON"

# 5) OFF = in [start-3, start), but do NOT overwrite ON
def apply_off_labels(df, tasks):
    # Work per (video,user) for speed and clarity
    for (vn, uid), tgrp in tasks.groupby(["Video Name (folder)","User ID (file)"]):
        sel = (df["Video Name (folder)"] == vn) & (df["User ID (file)"] == uid)
        if not sel.any():
            continue
        for _, r in tgrp.iterrows():
            pre_start = max(0.0, float(r.t_start) - 3.0)
            mask = sel & (df["TIME"] >= pre_start) & (df["TIME"] < r.t_start) & (df["label"] != "ON")
            df.loc[mask, "label"] = "OFF"
    return df

fix_m = apply_off_labels(fix_m, tasks)
gaz_m = apply_off_labels(gaz_m, tasks)

# 6) Keep your original columns + new label
keep_cols = list(fixations_df.columns) + ["label"]
fixations_labeled = fix_m[keep_cols].copy()
gaze_labeled      = gaz_m[list(gaze_df.columns) + ["label"]].copy()

# (optional) save
fixations_labeled.to_csv("outputs/fixations_with_on_off.csv", index=False)

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_50738/2988869100.py:3: DtypeWarning: Columns (14,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  gaze_df      = pd.read_csv("outputs/gaze_all_rows.csv")
